# NB02: Data Preparation

This notebook turns the raw match JSON saved by NB01 into one tidy table, with one row per team per match, ready for NB03 to explore without repeating this transformation.

In [1]:
import json
from pathlib import Path

import pandas as pd

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

COMPETITIONS = {
    "PL": "Premier League",
    "PD": "La Liga",
    "BL1": "Bundesliga",
    "SA": "Serie A",
    "FL1": "Ligue 1",
}
SEASON = 2025

## From one row per match to one row per team-match

To compare home and away performance I need a row per team per match rather than a row per match: each finished fixture becomes two rows, one for the home team and one for the away team, each carrying its own venue, goals for/against and points earned. This is the shape NB03 needs to compute home vs away averages per team and per league.

In [2]:
def result_and_points(goals_for, goals_against):
    if goals_for > goals_against:
        return "win", 3
    if goals_for == goals_against:
        return "draw", 1
    return "loss", 0


def build_team_rows(payload, league_code, league_name):
    rows = []
    for match in payload["matches"]:
        full_time = match["score"]["fullTime"]
        home_goals, away_goals = full_time["home"], full_time["away"]
        if home_goals is None or away_goals is None:
            continue  # skip any match without a final score

        home_result, home_points = result_and_points(home_goals, away_goals)
        away_result, away_points = result_and_points(away_goals, home_goals)

        base = {
            "league": league_name,
            "league_code": league_code,
            "season": SEASON,
            "match_id": match["id"],
            "date": match["utcDate"][:10],
            "matchday": match["matchday"],
        }
        rows.append({
            **base,
            "team": match["homeTeam"]["name"],
            "opponent": match["awayTeam"]["name"],
            "venue": "home",
            "goals_for": home_goals,
            "goals_against": away_goals,
            "goal_diff": home_goals - away_goals,
            "result": home_result,
            "points": home_points,
        })
        rows.append({
            **base,
            "team": match["awayTeam"]["name"],
            "opponent": match["homeTeam"]["name"],
            "venue": "away",
            "goals_for": away_goals,
            "goals_against": home_goals,
            "goal_diff": away_goals - home_goals,
            "result": away_result,
            "points": away_points,
        })
    return rows


all_rows = []
for code, name in COMPETITIONS.items():
    path = RAW_DIR / f"{code}_matches_{SEASON}.json"
    with open(path, encoding="utf-8") as f:
        payload = json.load(f)
    all_rows.extend(build_team_rows(payload, code, name))

matches_team_level = pd.DataFrame(all_rows)
matches_team_level.head()

,league,league_code,season,match_id,date,matchday,team,opponent,venue,goals_for,goals_against,goal_diff,result,points
0,Premier League,PL,2025,537785,2025-08-15,1,Liverpool FC,AFC Bournemouth,home,4,2,2,win,3
1,Premier League,PL,2025,537785,2025-08-15,1,AFC Bournemouth,Liverpool FC,away,2,4,-2,loss,0
2,Premier League,PL,2025,537786,2025-08-16,1,Aston Villa FC,Newcastle United FC,home,0,0,0,draw,1
3,Premier League,PL,2025,537786,2025-08-16,1,Newcastle United FC,Aston Villa FC,away,0,0,0,draw,1
4,Premier League,PL,2025,537787,2025-08-16,1,Brighton & Hove Albion FC,Fulham FC,home,1,1,0,draw,1


## Sanity checks

Two checks before saving: every raw match should produce exactly two rows (home and away), and each `match_id` should have exactly one "home" row and one "away" row.

In [3]:
raw_match_counts = {}
for code in COMPETITIONS:
    path = RAW_DIR / f"{code}_matches_{SEASON}.json"
    with open(path, encoding="utf-8") as f:
        raw_match_counts[code] = len(json.load(f)["matches"])

expected_rows = sum(raw_match_counts.values()) * 2
print(f"Expected rows: {expected_rows}, got: {len(matches_team_level)}")
assert len(matches_team_level) == expected_rows

venues_per_match = matches_team_level.groupby("match_id")["venue"].apply(lambda v: sorted(v))
assert venues_per_match.apply(lambda v: v == ["away", "home"]).all()

print(matches_team_level["result"].value_counts())
print(matches_team_level.groupby("league")["team"].nunique())

Expected rows: 3502, got: 3502
result
win     1306
loss    1306
draw     890
Name: count, dtype: int64
league
Bundesliga        18
La Liga           20
Ligue 1           18
Premier League    20
Serie A           20
Name: team, dtype: int64


## Saving the tidy table

In [4]:
out_path = PROCESSED_DIR / "matches_team_level.csv"
matches_team_level.to_csv(out_path, index=False)
print(f"Saved {len(matches_team_level)} rows to {out_path}")

Saved 3502 rows to data\processed\matches_team_level.csv
